# QLoRA on Qwen2.5-1.5B-Instruct: Colab T4

Runs the repo's own scripts end to end:

1. `src/prepare_data.py`: writes `data/train.jsonl` (21 rows) and `data/holdout.jsonl` (9 rows)
2. `src/train_lora.py`: 4-bit NF4 base + LoRA r=8 on `q_proj`/`v_proj`, saves the adapter to `outputs/lora`
3. `src/predict.py`: base (adapter off) vs LoRA predictions on the holdout, writes `data/preds.jsonl` with `label`, `base`, `lora`
4. `src/compare.py`: prints accuracy for both columns

Runtime → Change runtime type → **T4 GPU** before running.

The holdout has 9 rows. Treat any accuracy from it as a smoke test, not a benchmark.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

Get the repo. Set `REPO_URL` to your GitHub repo, or upload the folder and skip the clone.

In [ ]:
REPO_URL = "https://github.com/YOUR_GITHUB_USER/lora-qwen-serve.git"  # <- edit
import os
if not os.path.exists("lora-qwen-serve"):
    !git clone {REPO_URL} lora-qwen-serve
%cd lora-qwen-serve

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!python src/prepare_data.py

Train. On a T4 this is expected to take minutes, not hours, but that has not been timed from this repo yet.

In [ ]:
!python src/train_lora.py --epochs 3 --out outputs/lora

In [ ]:
# Adapter files should exist: adapter_config.json, adapter_model.safetensors, train_log.json
!ls -la outputs/lora

In [ ]:
!python src/predict.py --adapter outputs/lora --holdout data/holdout.jsonl --out data/preds.jsonl

In [ ]:
!head -n 3 data/preds.jsonl
!python src/compare.py --preds data/preds.jsonl

Save the results. Download the zip, then commit `data/preds.jsonl` and `outputs/lora/train_log.json` to the repo.
Paste the `compare.py` output into the README unedited. Keep adapter weights out of git, or upload them to the HF Hub.

In [ ]:
!zip -qr lora_run.zip outputs/lora data/preds.jsonl
from google.colab import files
files.download("lora_run.zip")